# Session 06 – Data Cleaning (Part 1: Missing Values)

This notebook creates a fictional Gujarat two-wheeler marketplace dataset and demonstrates missing-value identification, missingness mechanisms, dropping strategies, and filling strategies.

**Random seed:** 42


In [ ]:
import numpy as np
import pandas as pd


## Task 1 – Generate 400 raw bike listings

In [ ]:
# Generate the raw two-wheeler listings with seed 42
rng = np.random.default_rng(42)
n = 400

brands = ["Veloce", "Rajhans", "Sundar", "Tarang"]
cities = ["Ahmedabad", "Surat", "Vadodara", "Rajkot"]

bikes = pd.DataFrame({
    "listing_id": [f"BIKE{i:03d}" for i in range(1, n + 1)],
    "brand": rng.choice(brands, n),
    "city": rng.choice(cities, n),
    "model_year": rng.integers(2008, 2024, n),
    "km_driven": rng.integers(2000, 90001, n),
    "owner_count": rng.integers(1, 4, n),
    "price_inr": rng.integers(25000, 140001, n),
})

# Make exactly 8% of km_driven values missing
km_missing_idx = rng.choice(bikes.index, size=32, replace=False)
bikes.loc[km_missing_idx, "km_driven"] = np.nan

# Make half of owner_count values missing for model years below 2013
old_owner_idx = bikes.index[bikes["model_year"] < 2013].to_numpy()
rng.shuffle(old_owner_idx)
owner_missing_idx = old_owner_idx[:len(old_owner_idx) // 2]
bikes.loc[owner_missing_idx, "owner_count"] = np.nan

# Make the top 5% of price_inr values missing using the 95th percentile
price_cutoff = bikes["price_inr"].quantile(0.95)
price_missing_idx = bikes.index[bikes["price_inr"] >= price_cutoff]
bikes.loc[price_missing_idx, "price_inr"] = np.nan

bikes.to_csv("bike_listings_raw.csv", index=False)
print("Raw dataset shape:", bikes.shape)
print("Price 95th percentile cut-off:", price_cutoff)


## Task 2 – Missing-value report and missingness mechanisms

In [ ]:
# Count and percentage of missing values in every column
missing_report = pd.DataFrame({
    "column": bikes.columns,
    "missing_count": bikes.isna().sum().values,
    "missing_percentage": (bikes.isna().mean() * 100).round(2).values,
})

# Add the conceptual missingness mechanism for the columns specified
mechanisms = {
    "km_driven": "MCAR",
    "owner_count": "MAR",
    "price_inr": "MNAR",
}
missing_report["mechanism"] = missing_report["column"].map(mechanisms).fillna("No missing values")

# Number of fully complete rows using notnull()
complete_rows = bikes.notnull().all(axis=1).sum()

print(missing_report)
print("Fully complete rows:", complete_rows)

# Compare owner_count missing rates for older and newer model years
old_years = bikes["model_year"] < 2013
new_years = bikes["model_year"] >= 2013

old_missing_rate = bikes.loc[old_years, "owner_count"].isna().mean() * 100
new_missing_rate = bikes.loc[new_years, "owner_count"].isna().mean() * 100

print(f"Owner-count missing rate below 2013: {old_missing_rate:.2f}%")
print(f"Owner-count missing rate for 2013 and later: {new_missing_rate:.2f}%")

missing_report.to_csv("missing_report.csv", index=False)


### Missingness mechanisms – conceptual notes
- **MCAR:** Missingness is unrelated to observed or unobserved data.
- **MAR:** Missingness is related to another observed variable; here owner-count missingness depends on model year.
- **MNAR:** Missingness depends on the value itself; here the highest prices are intentionally hidden.

## Task 3 – Drop missing rows and columns

In [ ]:
# Drop rows that contain any missing value
drop_any = bikes.dropna(how="any")

# Keep rows with at least six non-missing values
drop_thresh = bikes.dropna(thresh=6)

# Keep columns with at least 360 non-missing values
drop_cols = bikes.dropna(axis=1, thresh=360)

comparison = pd.DataFrame({
    "dataset": ["raw", "drop_any", "drop_thresh"],
    "row_count": [len(bikes), len(drop_any), len(drop_thresh)],
    "column_count": [bikes.shape[1], drop_any.shape[1], drop_thresh.shape[1]],
    "mean_price_inr": [
        bikes["price_inr"].mean(),
        drop_any["price_inr"].mean(),
        drop_thresh["price_inr"].mean(),
    ],
})

print(comparison)
print("drop_cols shape:", drop_cols.shape)


## Task 4 – Fill missing values and save the cleaned dataset

In [ ]:
bikes_filled = bikes.copy()

# Fill km_driven with the median
bikes_filled["km_driven"] = bikes_filled["km_driven"].fillna(
    bikes_filled["km_driven"].median()
)

# Fill owner_count with the mode
bikes_filled["owner_count"] = bikes_filled["owner_count"].fillna(
    bikes_filled["owner_count"].mode()[0]
)

# Sort by model_year, then forward-fill and backward-fill price_inr
bikes_filled = bikes_filled.sort_values("model_year")
bikes_filled["price_inr"] = bikes_filled["price_inr"].ffill().bfill()
bikes_filled = bikes_filled.reset_index(drop=True)

# Compare mean prices for the requested datasets
print("Mean price – raw non-missing:", bikes["price_inr"].mean())
print("Mean price – drop_any:", drop_any["price_inr"].mean())
print("Mean price – bikes_filled:", bikes_filled["price_inr"].mean())

# Confirm that all missing values have been handled
print("Missing values after filling:")
print(bikes_filled.isna().sum())

assert bikes_filled.isna().sum().sum() == 0

bikes_filled.to_csv("bike_listings_s6.csv", index=False)
print("Cleaned dataset saved as bike_listings_s6.csv")


## Final checks

- Raw listings contain 400 rows.
- The missing-value report covers every raw column.
- `drop_any`, `drop_thresh`, and `drop_cols` use the required `dropna()` parameters.
- The cleaned dataset contains zero missing values and is saved to `bike_listings_s6.csv`.
